# 1. A first fit: the synthetic biphenyl

The bundled example is **synthetic, with a known answer**: 14 dipolar couplings of biphenyl
computed at the inter-ring torsion **φ = 37°** (rigid model, D2 symmetry), with noise of
σ = 0.5 Hz. A fit should find φ within a few standard errors of 37°. Its 68.3 % interval covers
the true value in about 68 % of noise realizations (the coverage Dipolar Studio was calibrated
on), so this fixed realization may or may not.

Everything below goes through `Studio`, the Python API of Dipolar Studio: each `call` is one
registry operation, the same the browser app sends.

In [1]:
from dipolar_app.api import Studio

studio = Studio.local()  # a session in this process (give it a directory to keep the project)
opened = studio.call("project.open_example", name="biphenyl-synthetic", expected_science_revision=0)
project = studio.call("project.describe")
print(project["name"], "-", len(project["couplings"]), "couplings,", len(project["spins"]), "protons")
print("model:", project["model"]["kind"], "on", [d["value"] for d in project["model"]["phi_domain"]], "deg")

Synthetic biphenyl (known answer) - 14 couplings, 10 protons
model: rigid on [0.0, 90.0] deg


## Preview: the residuals at a chosen torsion

`fit.preview` re-solves the order parameters at a fixed φ: nothing is recorded. The landscape
samples it over the torsion domain; its lowest sample is near the answer.

In [2]:
preview = studio.call("fit.preview", phi={"value": 37.0, "unit": "deg"})
print(f"at phi = 37 deg: chi2 = {preview['chi2']:.3f}")
landscape = studio.call("fit.landscape")
lowest = min((c, p) for c, p in zip(landscape["chi2"], landscape["phi"]) if c is not None)
print(f"landscape: {len(landscape['phi'])} samples, lowest chi2 = {lowest[0]:.3f} at phi = {lowest[1]:.1f} deg")

at phi = 37 deg: chi2 = 15.763
landscape: 181 samples, lowest chi2 = 15.763 at phi = 37.0 deg


## The fit, with profile intervals

`studio.fit()` submits `fit.run` as a job, waits for it and returns the run's result.

In [3]:
run = studio.fit(intervals=True)
theta = {t["name"]: t["value"] for t in run["theta"]}
order = {o["name"]: o["value"] for o in run["order"]}
se = {s["name"]: s["value"] for s in run["stderr"]}
print(f"phi   = {theta['phi_deg']:.4f} +/- {se['phi_deg']:.4f} deg")
print(f"S_zz  = {order['Szz']:.5f} +/- {se['Szz']:.5f}")
print(f"Delta = {order['Delta']:.5f} +/- {se['Delta']:.5f}")
print(f"chi2 = {run['chi2']:.2f} with nu = {run['nu']}, p = {run['p_value']:.3f}; adequate: {run['adequate']}")
for i in run["intervals"]:
    print(f"  {i['parameter']:8} [{i['lower']:.4f}, {i['upper']:.4f}]  supported: {i['supported']}")

phi   = 36.9850 +/- 0.0082 deg
S_zz  = 0.25004 +/- 0.00006
Delta = 0.03053 +/- 0.00020
chi2 = 12.43 with nu = 11, p = 0.332; adequate: True
  phi_deg  [36.9768, 36.9932]  supported: True
  Szz      [0.2500, 0.2501]  supported: True
  Delta    [0.0303, 0.0307]  supported: True


In [4]:
# The known answer: within three standard errors of phi = 37 deg.
phi_interval = next(i for i in run["intervals"] if i["parameter"] == "phi_deg")
deviation = (theta["phi_deg"] - 37.0) / se["phi_deg"]
print(f"phi - 37 deg = {deviation:+.2f} standard errors; the 68.3 % interval covers 37 deg:",
      phi_interval["lower"] < 37.0 < phi_interval["upper"])
assert run["validity"]["converged"] and run["validity"]["regular"] and run["adequate"]
assert phi_interval["supported"] and abs(deviation) < 3

phi - 37 deg = -1.83 standard errors; the 68.3 % interval covers 37 deg: False


## The report

`run.report` gives the fit as a self-contained HTML page and a CSV of the couplings. Save them
to keep or share the result.

In [5]:
run_id = studio.call("render.describe_view")["run"]["run_id"]
report = studio.call("run.report", run_id=run_id)
couplings_in_csv = len(report["csv"].splitlines()) - 1
print(report["filename"], "-", len(report["html"]), "bytes of HTML,", couplings_in_csv, "couplings in the CSV")
print(report["csv"].splitlines()[0])
studio.close()

dipolar-report-run_000001 - 9036 bytes of HTML, 14 couplings in the CSV
coupling_id,label,observed_Hz,sigma_Hz,calculated_Hz,residual_Hz,normalized_residual
